<a href="https://colab.research.google.com/github/sonia73b/tech405asst/blob/main/TECH405W6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
SOS_token = 0
EOS_token = 1

pairs = [
    ["i am happy", "म खुशी छु"],
    ["i am hungry", "म भोकाएको छु"],
    ["he is running", "ऊ दौडिरहेको छ"],
    ["she is singing", "उनी गीत गाउँदै छिन्"],
    ["we are learning", "हामी सिक्दैछौं"],
    ["they are playing", "उनीहरू खेलिरहेका छन्"],
    ["good morning", "शुभ प्रभात"],
    ["good night", "शुभ रात्री"],
    ["i love music", "म संगीत मन पराउँछु"],
    ["open the door", "ढोका खोल"],
    ["close the window", "झ्याल बन्द गर"],
    ["how are you", "तिमीलाई कस्तो छ"],
    ["i am fine", "म ठिक छु"],
    ["thank you", "धन्यवाद"],
    ["see you later", "पछि भेटौंला"]
]

for p in pairs[:5]:
    print(p)

['i am happy', 'म खुशी छु']
['i am hungry', 'म भोकाएको छु']
['he is running', 'ऊ दौडिरहेको छ']
['she is singing', 'उनी गीत गाउँदै छिन्']
['we are learning', 'हामी सिक्दैछौं']


In [3]:
class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2

    def add_sentence(self, sentence):
        for word in sentence.split(" "):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [4]:
input_lang = Lang("English")
output_lang = Lang("Nepali")

for eng, nep in pairs:
    input_lang.add_sentence(eng.lower())
    output_lang.add_sentence(nep)

print("Input language words:", input_lang.n_words)
print("Output language words:", output_lang.n_words)

Input language words: 32
Output language words: 35


In [5]:
def indexes_from_sentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(" ")]

def tensor_from_sentence(lang, sentence):
    indexes = indexes_from_sentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

def tensors_from_pair(pair):
    input_tensor = tensor_from_sentence(input_lang, pair[0].lower())
    target_tensor = tensor_from_sentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

In [6]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input_token, hidden):
        embedded = self.embedding(input_token).view(1, 1, -1)
        output = embedded
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def init_hidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [7]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_token, hidden):
        output = self.embedding(input_token).view(1, 1, -1)
        output = torch.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

In [8]:
teacher_forcing_ratio = 0.5

def train(input_tensor, target_tensor, encoder, decoder,
          encoder_optimizer, decoder_optimizer, criterion,
          max_length=20):

    encoder_hidden = encoder.init_hidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    loss = 0

    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    if use_teacher_forcing:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]
    else:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach().view(1, 1)

            loss += criterion(decoder_output, target_tensor[di])

            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

In [9]:
def train_iters(encoder, decoder, n_iters=5000, print_every=500, learning_rate=0.01):
    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)

    training_pairs = [tensors_from_pair(random.choice(pairs)) for _ in range(n_iters)]
    criterion = nn.NLLLoss()

    print("Training started...\n")

    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]

        loss = train(input_tensor, target_tensor, encoder, decoder,
                     encoder_optimizer, decoder_optimizer, criterion)

        if iter % print_every == 0:
            print(f"Iteration {iter}/{n_iters} - Loss: {loss:.4f}")

In [10]:
hidden_size = 128

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train_iters(encoder, decoder, n_iters=5000, print_every=500)

Training started...

Iteration 500/5000 - Loss: 2.1103
Iteration 1000/5000 - Loss: 1.5443
Iteration 1500/5000 - Loss: 0.0628
Iteration 2000/5000 - Loss: 0.0309
Iteration 2500/5000 - Loss: 0.0236
Iteration 3000/5000 - Loss: 0.0215
Iteration 3500/5000 - Loss: 0.0134
Iteration 4000/5000 - Loss: 0.0087
Iteration 4500/5000 - Loss: 0.0087
Iteration 5000/5000 - Loss: 0.0077


In [11]:
def evaluate(encoder, decoder, sentence, max_length=20):
    with torch.no_grad():
        input_tensor = tensor_from_sentence(input_lang, sentence.lower())
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.init_hidden()

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden

        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)

            if topi.item() == EOS_token:
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach().view(1, 1)

        return " ".join(decoded_words)

In [12]:
test_sentences = [
    "i am happy",
    "good morning",
    "thank you",
    "open the door",
    "i love music"
]

for sentence in test_sentences:
    output = evaluate(encoder, decoder, sentence)
    print("English :", sentence)
    print("Nepali  :", output)
    print()

English : i am happy
Nepali  : म खुशी छु

English : good morning
Nepali  : शुभ प्रभात

English : thank you
Nepali  : धन्यवाद

English : open the door
Nepali  : ढोका खोल

English : i love music
Nepali  : म संगीत मन पराउँछु

